# Baseline — generacja apelacji

Najnaiwniejsze podejście: **wszystkie akta w jeden prompt** + „napisz apelację”.
Ten notebook tylko **generuje** apelację (i liczy koszt). Ewaluację pokazuje
osobny notebook `eval_walkthrough.ipynb`.

> Notebook używa taniego `gpt-5.4-mini` (demo). W `__main__` modułu (`python -m baseline.main`) generacja i ocena idą modelem z `.env` (`gpt-5.4`).

## 0. Konfiguracja

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()
os.environ["LLM_MODEL"] = "gpt-5.4-mini"  # w notebookach tani model — to tylko demo
print("model:", os.environ["LLM_MODEL"], "| klucz z .env:", bool(os.environ.get("LLM_API_KEY")))

## 1. Wczytanie akt i rozmiar promptu

Baseline wrzuca **całość** w jeden prompt — policzmy, ile to tokenów.

In [ ]:
from src.loader import load_all
from src.tokens import count_tokens
from baseline.main import build_context
from baseline.prompts import SYSTEM_PROMPT

docs = load_all()
context = build_context(docs)
prompt_tokens = count_tokens(SYSTEM_PROMPT + "\n\n" + context)
print(f"Dokumentów: {len(docs)}")
print(f"Znaków w kontekście: {len(context):,}")
print(f"Tokenów w promptcie: ~{prompt_tokens:,}")

## 2. Generowanie apelacji (jeden prompt) + koszt

In [ ]:
from baseline.main import generate_appeal
from src.llm import track_usage
from src.cost import cost_summary

with track_usage() as u:
    appeal = generate_appeal(docs).tekst

print(f"Apelacja: {len(appeal):,} znaków")
print(f"Koszt generacji: {cost_summary(u, os.environ['LLM_MODEL'])}\n")
print(appeal[:1000], "...")

## 3. Zapis apelacji

Do `data/output/` (nazwa: podejście + znacznik czasu; poza gitem).

In [ ]:
from src.output import save_appeal

path = save_appeal(appeal, "baseline")
print("Zapisano:", path)

## Podsumowanie

Baseline = **1 wywołanie LLM** z ogromnym promptem. Szybkie, ale: zapchany kontekst, „lost in the middle”, brak śladu rozumowania, podatność na pominięcia.

Jak dobra jest ta apelacja? → `eval_walkthrough.ipynb` (pokrycie + jakość).